In [3]:
import csv
import warnings

import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib as mpl

from typing import List
from scipy import integrate
from datetime import datetime, time, timedelta
from matplotlib import pyplot as plt
from matplotlib import font_manager as fm


warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.display.float_format = '.{:.5f}'.format
sns.reset_defaults()
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Linux Libertine"],
    "font.size": 9,
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{libertine}",
    "mathtext.fontset": "custom",
    "mathtext.rm": "Linux Libertine",
    "mathtext.it": "Linux Libertine:italic",
    "mathtext.bf": "Linux Libertine:bold",
})

FIGSIZE_ACM_SMALL = (3.33, 2.06)
FIGSIZE_ACM_SMALL = (7.00, 2.06)

wearables = [[86, 166, 211], [26, 63, 110], [87, 132, 148], [224, 88, 108], [16, 80, 103]]
wearables_palette = [[r/255, g/255, b/255] for r, g, b in wearables ]
sns.set_palette(wearables_palette)

In [12]:
def get_csv_as_list(filename: str) -> list[list[str]]:
    values = []
    with open(filename) as f:
        reader = csv.reader(f)
        for row in reader:
            row[0] = row[0][20:]
            values.append(row)
    return values[1:]

def parse_csv(filename: str) -> pd.DataFrame:
    def parse(data: list[str]) -> pd.DataFrame:
            return pd.DataFrame(data, columns=["message_id", "t_produced", "t_observed"])
    data = get_csv_as_list(filename)
    return parse(data)

def transform_columns_to_numerics(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["message_id"] = pd.to_numeric(df["message_id"])
    df["t_produced"] = pd.to_numeric(df["t_produced"])
    df["t_observed"] = pd.to_numeric(df["t_observed"])
    return df

def filter_errors(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    

def reduce_frame(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    reduced_df = df.groupby(["message_id"], as_index=True).agg(
        t_produced=("t_produced", "min"), 
        t_observed=("t_observed", "max")
    )
    return reduced_df


def trim_benchmark(df: pd.DataFrame, ramp_up_s: int = 15, ramp_down_s: int = 5) -> pd.DataFrame:
    df = df.copy()
    start_time = df["t_produced"].min() + ramp_up_s * 1_000_000_000
    end_time = df["t_observed"].max() - ramp_down_s * 1_000_000_000

    trimmed = df[(df["t_produced"] >= start_time) & (df["t_observed"] <= end_time)]
    return trimmed

def median_processing_time(df: pd.DataFrame) -> float:
    df = df.copy()
    df["processing_time"] = df["t_observed"] - df["t_produced"]
    median_pt = df["processing_time"].median()
    return median_pt

def convert_to_sec(nano: float) -> float:
    return nano / (1_000_000_000)

def process(filename: str) -> float:
    csv = parse_csv(filename)
    transformed = transform_columns_to_numerics(csv)
    # trimmed = trim_benchmark(transformed)
    reduced = reduce_frame(transformed)
    return convert_to_sec(median_processing_time(reduced))

def average_median(m1: float, m2: float, m3: float) -> float:
    avg = (m1 + m2 + m3) / 3
    return avg

In [5]:
# Mapper

# 1KB, 1RPS, 60s
median01_1kb = process("measurements/mapper/1kb_1RPS_60s_01.csv")
median02_1kb = process("measurements/mapper/1kb_1RPS_60s_02.csv")
median03_1kb = process("measurements/mapper/1kb_1RPS_60s_03.csv")


# 10 KB, 1RPS, 60s
median01_10kb = process("measurements/mapper/10kb_1RPS_60s_01.csv")
median02_10kb = process("measurements/mapper/10kb_1RPS_60s_02.csv")
median03_10kb = process("measurements/mapper/10kb_1RPS_60s_03.csv")

# 50 KB, 1RPS, 60s
median01_50kb = process("measurements/mapper/50kb_1RPS_60s_01.csv")
median02_50kb = process("measurements/mapper/50kb_1RPS_60s_02.csv")
median03_50kb = process("measurements/mapper/50kb_1RPS_60s_03.csv")

# 25KB, 1RPS, 60s
median01_25kb = process("measurements/mapper/25kb_1RPS_60s_01.csv")
median02_25kb = process("measurements/mapper/25kb_1RPS_60s_02.csv")
median03_25kb = process("measurements/mapper/25kb_1RPS_60s_03.csv")
print(median01_25kb)
print(median02_25kb)
print(median03_25kb)

print("mapper 1kb:")
print(f"{average_median(median01_1kb, median02_1kb, median03_1kb)}s")
print("mapper 10kb:")
print(f"{average_median(median01_10kb, median02_10kb, median03_10kb)}s")
print("mapper 25kb:")
print(f"{average_median(median01_25kb, median02_25kb, median03_25kb)}s")
print("mapper 50kb:")
print(f"{average_median(median01_50kb, median02_50kb, median03_50kb)}s")

# new mapper
median01_1kb_new = process("measurements/mapper_new/1kb_1RPS_60s_01.csv")
median02_1kb_new = process("measurements/mapper_new/1kb_1RPS_60s_02.csv")
median03_1kb_new = process("measurements/mapper_new/1kb_1RPS_60s_03.csv")

median01_10kb_new = process("measurements/mapper_new/10kb_1RPS_60s_01.csv")
median02_10kb_new = process("measurements/mapper_new/10kb_1RPS_60s_02.csv")
median03_10kb_new = process("measurements/mapper_new/10kb_1RPS_60s_03.csv")

median01_25kb_new = process("measurements/mapper_new/25kb_1RPS_60s_01.csv")
median02_25kb_new = process("measurements/mapper_new/25kb_1RPS_60s_02.csv")
median03_25kb_new = process("measurements/mapper_new/25kb_1RPS_60s_03.csv")

median01_50kb_new = process("measurements/mapper_new/50kb_1RPS_60s_01.csv")
median02_50kb_new = process("measurements/mapper_new/50kb_1RPS_60s_02.csv")
median03_50kb_new = process("measurements/mapper_new/50kb_1RPS_60s_03.csv")

print("new mapper 1kb:")
print(f"{average_median(median01_1kb_new, median02_1kb_new, median03_1kb_new)}s")
print("new mapper 10kb:")
print(f"{average_median(median01_10kb_new, median02_10kb_new, median03_10kb_new)}s")
print("new mapper 25kb:")
print(f"{average_median(median01_25kb_new, median02_25kb_new, median03_25kb_new)}s")
print("new mapper 50kb:")
print(f"{average_median(median01_50kb_new, median02_50kb_new, median03_50kb_new)}s")

# sampled validation mapper
median01_1kb_sampling = process("measurements/mapper_sampling/1kb_1RPS_60s_01.csv")
median02_1kb_sampling = process("measurements/mapper_sampling/1kb_1RPS_60s_02.csv")
median03_1kb_sampling = process("measurements/mapper_sampling/1kb_1RPS_60s_03.csv")

median01_10kb_sampling = process("measurements/mapper_sampling/10kb_1RPS_60s_01.csv")
median02_10kb_sampling = process("measurements/mapper_sampling/10kb_1RPS_60s_02.csv")
median03_10kb_sampling = process("measurements/mapper_sampling/10kb_1RPS_60s_03.csv")

median01_25kb_sampling = process("measurements/mapper_sampling/25kb_1RPS_60s_01.csv")
median02_25kb_sampling = process("measurements/mapper_sampling/25kb_1RPS_60s_02.csv")
median03_25kb_sampling = process("measurements/mapper_sampling/25kb_1RPS_60s_03.csv")

median01_50kb_sampling = process("measurements/mapper_sampling/50kb_1RPS_60s_01.csv")
median02_50kb_sampling = process("measurements/mapper_sampling/50kb_1RPS_60s_02.csv")
median03_50kb_sampling = process("measurements/mapper_sampling/50kb_1RPS_60s_03.csv")

print("sampling mapper 1kb:")
print(f"{average_median(median01_1kb_sampling, median02_1kb_sampling, median03_1kb_sampling)}")
print("sampling mapper 10kb:")
print(f"{average_median(median01_10kb_sampling, median02_10kb_sampling, median03_10kb_sampling)}")
print("sampling mapper 25kb:")
print(f"{average_median(median01_25kb_sampling, median02_25kb_sampling, median03_25kb_sampling)}")
print("sampling mapper 50kb:")
print(f"{average_median(median01_50kb_sampling, median02_50kb_sampling, median03_50kb_sampling)}")


8.635576276
9.056472492
11.627568411
mapper 1kb:
0.21188584533333332s
mapper 10kb:
1.5661300413333332s
mapper 25kb:
9.773205726333332s
mapper 50kb:
20.102922239999998s
new mapper 1kb:
0.2323306026666667s
new mapper 10kb:
0.9026520036666668s
new mapper 25kb:
2.0129730670000003s
new mapper 50kb:
4.058866907333333s
sampling mapper 1kb:
0.21565281666666666
sampling mapper 10kb:
0.2170477915
sampling mapper 25kb:
0.18373133983333334
sampling mapper 50kb:
0.22580457483333335


In [9]:
# RPS effects on processing time:


plot3r = pd.DataFrame ({
    "Requests/Second": [1, 2, 3],
    "Median Latency": [34.336, 36.22, 38.333] # TODO -> Test values
})

plot5r = pd.DataFrame ({
    "Requests/Second": [1, 2, 3],
    "Median Latency": [32.12, 33.5, 35.99] 
})

plot7r = pd.DataFrame ({
    "Requests/Second": [1, 2, 3],
    "Median Latency": [31.00, 32.8, 35.7] 
})

df = pd.concat([
    plot3r.assign(system="3 Replicas"),
    plot5r.assign(system="5 Replicas"),
    plot7r.assign(system="7 Replicas"),
])

fig, ax = plt.subplots(figsize=FIGSIZE_ACM_SMALL, constrained_layout=True)

sns.lineplot(
    data=df,
    x="Requests/Second",
    y="Median Latency",
    hue="system",
    style="system",
    markers=True,
    dashes=True,
    ax=ax
)

ax.set_ylabel("Median Latency [s]")
ax.set_xlabel("Requests/Second")
ax.set_ylim(30, 45)
ax.set_xlim(0.5, 3.5)
ax.set_xticks([1,2,3])
ax.set_xticklabels(["$1$", "$10$", "$100$"])

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=handles, labels=labels, loc="center left")

fig.savefig("plots/rps_impact_on_processing_time.pdf", bbox_inches=False)
plt.close()

In [44]:
# Message-Size Impact on Processing Time

plot2 = pd.DataFrame({
    "MessageSize": [1, 2, 3, 4],
    "ProcessingTime": [0.23223270733333334, 0.90116695, 2.0129730670000003, 4.058866907333333]
}).assign(Processing="Not Sampled")

plot3 = pd.DataFrame({
    "MessageSize": [1, 2, 3, 4],
    "ProcessingTime": [0.21565281666666666, 0.2170477915, 0.18373133983333334, 0.22580457483333335]
}).assign(Processing="Sampled")

plot = pd.concat([plot2, plot3])

fig, ax = plt.subplots(figsize=(FIGSIZE_ACM_SMALL[0], FIGSIZE_ACM_SMALL[1]*1.5), constrained_layout=True)

sns.lineplot(
    data=plot,
    x="MessageSize",
    y="ProcessingTime",
    hue="Processing",
    style="Processing",
    markers=True,
    dashes=True,
    ax=ax
)

ax.set_ylabel("Processing Time [s]")
ax.set_xlabel("Message Size [KB]")
ax.set_ylim(0, 6)
ax.set_xlim(0.5, 4.5)
ax.set_xticks([1,2,3,4])
ax.set_xticklabels(["$1$", "$10$", "$25$","$50$"])

fig.savefig("plots/message-size-impact.pdf", bbox_inches=False)
plt.close()

In [67]:
# Ingestion-Service

# What do I want to show?
# Has Message-Size an impact on Response Time?
# Where is the Breakpoint? -> For 10KB Messages (not tested yet)

# 1KB messages

def parse_csv_ingestion_service(filename: str) -> pd.DataFrame:
    def parse(data: list[str]) -> pd.DataFrame:
        return pd.DataFrame(data, columns=["message_id", "status_code", "t_send", "t_received"])
    data = get_csv_as_list(filename)
    return parse(data)

def transform_columns_to_numerics_is(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["message_id"] = pd.to_numeric(df["message_id"])
    df["status_code"] = pd.to_numeric(df["status_code"])
    df["t_send"] = pd.to_numeric(df["t_send"])
    df["t_received"] = pd.to_numeric(df["t_received"])
    return df

def trim_benchmark_is(df: pd.DataFrame, ramp_up_s: int = 15, ramp_down_s: int = 5) -> pd.DataFrame:
    df = df.copy()

    df["t_send"] = pd.to_numeric(df["t_send"])
    
    min_time = df['t_send'].min()
    max_time = df['t_send'].max()
    
    start_cutoff = min_time + ramp_up_s * 1_000_000_000
    end_cutoff = max_time - ramp_down_s * 1_000_000_000
    
    trimmed_df = df[
        (df['t_send'] >= start_cutoff) & 
        (df['t_send'] <= end_cutoff)
    ]
    
    return trimmed_df


def calculate_error_rate(df: pd.DataFrame) -> float:
    df = df.copy()

    error_count = (df["status_code"] != 200).sum()
    print(f"counted: {error_count} errors")

    total_rows = len(df)
    error_rate = error_count / total_rows if total_rows > 0 else 0
    
    return error_rate

def ingestion_service_benchmark_metrics(df: pd.DataFrame) -> dict:
    df["response_time"] = df["t_received"] - df["t_send"]
    
    successful = df[df["status_code"] == 200]
    errors = df[df["status_code"] != 200]

    median_response_time = ns_to_ms(successful["response_time"].median())
    p95_response_time = ns_to_ms(successful["response_time"].quantile(0.95))
    p99_response_time = ns_to_ms(successful["response_time"].quantile(0.99))
    error_rate = len(errors) / len(df) if len(df) > 0 else 0
    median_error_response_time =  ns_to_ms(errors["response_time"].median())


    print(f"median_response_time: {median_response_time}ms")
    print(f"p95_response_time: {p95_response_time}ms")
    print(f"p99_response_time: {p99_response_time}ms")
    print(f"error_rate: {error_rate}")
    if error_rate > 0:
        print(f"median_error_response_time: {median_error_response_time}ms")

def median_response_time_overtime(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["t_received"] = pd.to_numeric(df["t_received"])
    df["t_send"] = pd.to_numeric(df["t_send"])

    df["response_time"] = df["t_received"] - df["t_send"]

    t0 = df["t_received"].min()
    df["second"] = (df["t_received"] - t0) // 1_000_000_000

    ok = df[df["status_code"] == 200]
    result = (
        ok.groupby("second")["response_time"]
          .median()
          .reset_index(name="median_response_time_ns")
    )
    result["median_response_time_ms"] = result["median_response_time_ns"].apply(ns_to_ms)

    result["second"] = result["second"].astype(int)
    
    return result[["second", "median_response_time_ms"]]


def throughput_error_rate_per_s(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["t_received"] = pd.to_numeric(df["t_received"])
    
    t0 = df["t_received"].min()
    df["second"] = (df["t_received"] - t0) // 1_000_000_000

    ok = df[df["status_code"] == 200]
    err = df[df["status_code"] != 200]

    throughput = (
        ok.groupby("second")
          .size()
          .reset_index(name="throughput")
    )

    errs = (
        err.groupby("second")
           .size()
           .reset_index(name="errors")
    )

    result = pd.merge(throughput, errs, on="second", how="outer").fillna(0)

    result["throughput"] = result["throughput"].astype(int)
    result["errors"] = result["errors"].astype(int)

    return result.sort_values("second").reset_index(drop=True)


def process_ingestion_service(filename: str):
    df = parse_csv_ingestion_service(filename)
    df = trim_benchmark_is(df)
    df = transform_columns_to_numerics_is(df)
    # ingestion_service_benchmark_metrics(df)
    return throughput_error_rate_per_s(df)
    # return median_response_time_overtime(df)

def process_response_time(filename: str):
    df = parse_csv_ingestion_service(filename)
    df = trim_benchmark_is(df)
    df = transform_columns_to_numerics_is(df)
    return median_response_time_overtime(df)


def error_rate(filename: str) -> float:
    df = parse_csv_ingestion_service(filename)
    df = transform_columns_to_numerics_is(df)
    return calculate_error_rate(df)

def avg_of_columns(df1, df2, df3):
    return pd.concat([df1, df2, df3]).groupby(level=0).mean()
    

def ns_to_ms(nano: int) -> float:
    return nano / (1_000_000)

# process_ingestion_service("measurements/ingestion-service/1kb_1RPS_60s_01.csv")
# process_ingestion_service("measurements/ingestion-service/1kb_1RPS_60s_02.csv")
# process_ingestion_service("measurements/ingestion-service/1kb_1RPS_60s_03.csv")
print("50KB, 50RPS, 60s")
df1 = process_ingestion_service("measurements/ingestion-service/50kb_50RPS_60s_01.csv")
print("---")
df2 = process_ingestion_service("measurements/ingestion-service/50kb_50RPS_60s_02.csv")
print("---")
df3 = process_ingestion_service("measurements/ingestion-service/50kb_50RPS_60s_03.csv")

avg = avg_of_columns(df1, df2, df3)

# Plot
# avg has 2 curves

plotThroughput = avg[["second", "throughput"]].rename(columns={"throughput": "value"})
plotThroughput["Metric"] = "Throughput/s"

plotErrors = avg[["second", "errors"]].rename(columns={"errors": "value"})
plotErrors["Metric"] = "Errors/s"

plot = pd.concat([plotThroughput, plotErrors], ignore_index=True)

fig, ax = plt.subplots(figsize=FIGSIZE_ACM_SMALL, constrained_layout=True)

sns.lineplot(
    data=plot,
    x="second",
    y="value",
    hue="Metric",
    style="Metric",
    markers=True,
    dashes=True,
    ax=ax
)

ax.set_xlabel("Time [s]")
ax.set_ylabel("Count")
ax.set_ylim(0, max(plot["value"]) * 1.1)

fig.savefig("plots/ingestion-service-throughput-errors.pdf", bbox_inches=False)
plt.close()


# plot response time
df1 = process_response_time("measurements/ingestion-service/50kb_50RPS_60s_01.csv")
df2 = process_response_time("measurements/ingestion-service/50kb_50RPS_60s_02.csv")
df3 = process_response_time("measurements/ingestion-service/50kb_50RPS_60s_03.csv")

avg = avg_of_columns(df1, df2, df3)

print(avg)

plotResponseTime = avg[["second", "median_response_time_ms"]].rename(columns={"median_response_time_ms": "value"}).assign(Metric="Response Time")

fig, ax = plt.subplots(figsize=FIGSIZE_ACM_SMALL, constrained_layout=True)
sns.lineplot(
    data=plotResponseTime,
    x="second",
    y="value",
    hue="Metric",
    style="Metric",
    markers=True,
    dashes=True,
    ax=ax
)

ax.set_xlabel("Time [s]")
ax.set_ylabel("Median Response Time [ms]")
ax.set_ylim(0, 10)

fig.savefig("plots/ingestion-service-response-time.pdf", bbox_inches=False)
plt.close()


50KB, 50RPS, 60s
---
---
      second  median_response_time_ms
0   .0.00000                 .6.09063
1   .1.00000                 .6.22754
2   .2.00000                 .6.18560
3   .3.00000                 .6.18389
4   .4.00000                 .6.07332
5   .5.00000                 .6.30549
6   .6.00000                 .6.41503
7   .7.00000                 .6.26409
8   .8.00000                 .6.12704
9   .9.00000                 .6.32050
10 .10.00000                 .6.34399
11 .11.00000                 .6.28196
12 .12.00000                 .6.11726
13 .13.00000                 .6.24561
14 .14.00000                 .6.20858
15 .15.00000                 .6.34092
16 .16.00000                 .6.17181
17 .17.00000                 .6.14050
18 .18.00000                 .6.13724
19 .19.00000                 .6.13982
20 .20.00000                 .6.15199
21 .21.00000                 .6.24392
22 .22.00000                 .6.05245
23 .23.00000                 .6.27879
24 .24.00000             

In [15]:
# End-to-End-Processing-time

def parse_csv_endtoend_service(filename: str) -> pd.DataFrame:
    def parse(data: list[str]) -> pd.DataFrame:
        return pd.DataFrame(data, columns=["message-id", "t_produced", "t_response","t_ingested", "error"])
    data = get_csv_as_list(filename)
    return parse(data)


def median_response_time_overtime(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["t_ingested"] = pd.to_numeric(df["t_ingested"])
    df["t_produced"] = pd.to_numeric(df["t_produced"])

    df["processing_time"] = df["t_ingested"] - df["t_produced"]

    
    t0 = df["t_ingested"].min()
    tMax = df["t_ingested"].max()

    
    df["second"] = (df["t_ingested"] - t0) // 1_000

    result = (
        df.groupby("second")["processing_time"]
          .median()
          .reset_index(name="median_processing_time_ms")
    )

    result["second"] = result["second"].astype(int)
    return result[["second", "median_processing_time_ms"]]

def transform_columns_to_numerics_is(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["t_produced"] = pd.to_numeric(df["t_produced"])
    df["t_ingested"] = pd.to_numeric(df["t_ingested"])
    return df

def process_end_to_end_latency(filename: str) -> pd.DataFrame:
    df = parse_csv_endtoend_service(filename)
    return median_response_time_overtime(df)

def avg_of_columns(df1, df2, df3):
    return pd.concat([df1, df2, df3]).groupby(level=0).mean()

def ns_to_ms(nano: int) -> float:
    return nano / (1_000_000)


e2e1 = process_end_to_end_latency("measurements/end-to-end_processing_time/1KB_01.csv")
e2e2 = process_end_to_end_latency("measurements/end-to-end_processing_time/1KB_02.csv")
e2e3 = process_end_to_end_latency("measurements/end-to-end_processing_time/1KB_03.csv")

e2e10 = process_end_to_end_latency("measurements/end-to-end_processing_time/10KB_01.csv")
e2e20 = process_end_to_end_latency("measurements/end-to-end_processing_time/10KB_02.csv")
e2e30 = process_end_to_end_latency("measurements/end-to-end_processing_time/10KB_03.csv")

e2e100 = process_end_to_end_latency("measurements/end-to-end_processing_time/50KB_01.csv")
e2e200 = process_end_to_end_latency("measurements/end-to-end_processing_time/50KB_02.csv")
e2e300 = process_end_to_end_latency("measurements/end-to-end_processing_time/50KB_03.csv")


avg = avg_of_columns(e2e1, e2e2, e2e3)
avg2 = avg_of_columns(e2e10,e2e20,e2e30)
avg3 = avg_of_columns(e2e100,e2e200,e2e300)

print(avg["median_processing_time_ms"].mean())
print(avg2["median_processing_time_ms"].mean())
print(avg3["median_processing_time_ms"].median())

plotProcessingTime = avg[["second", "median_processing_time_ms"]].rename(columns={"median_processing_time_ms": "value"}).assign(Metric="Processing Time")


fig, ax = plt.subplots(figsize=FIGSIZE_ACM_SMALL, constrained_layout=True)
sns.lineplot(
    data=plotProcessingTime,
    x="second",
    y="value",
    hue="Metric",
    style="Metric",
    markers=True,
    dashes=True,
    ax=ax
)

ax.set_xlabel("Time [s]")
ax.set_ylabel("Median Processing Time [ms]")
ax.set_ylim(0, 1000)

fig.savefig("plots/end-to-end-processing-time.pdf", bbox_inches=False)
plt.close()


422.8611111111111
2047.0035460992908
12139.083333333332


In [193]:
# ML Pipeline Processing Time (Overtime)

def parse_csv_ml_service(filename: str) -> pd.DataFrame:
    def parse(data: list[str]) -> pd.DataFrame:
        return pd.DataFrame(data, columns=["message-id","t-produced","t-observed"])
    data = get_csv_as_list(filename)
    return parse(data)


def process_ml_processing_time(filename: str) -> pd.DataFrame:
    df = parse_csv_ml_service(filename)
    df = median_processing_time(df)
    return df

def median_processing_time(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["t-observed"] = pd.to_numeric(df["t-observed"])
    df["t-produced"] = pd.to_numeric(df["t-produced"])

    df["processing_time"] = df["t-observed"] - df["t-produced"]

    t0 = df["t-observed"].min()
    df["second"] = (df["t-observed"] - t0) // 1_000

    result = (
        df.groupby("second")["processing_time"]
        .median()
        .apply(lambda x: x // 1_000)
        .reset_index(name="median_processing_time_ms")
    )

    result["second"] = result["second"].astype(int)
    return result[["second", "median_processing_time_ms"]]

ml1 = process_ml_processing_time("measurements/ml-pipeline_processing_time/10RPS_60s_01.csv")
ml2 = process_ml_processing_time("measurements/ml-pipeline_processing_time/10RPS_60s_02.csv")
ml3 = process_ml_processing_time("measurements/ml-pipeline_processing_time/10RPS_60s_03.csv")

def avg_of_columns(df1, df2, df3):
    return pd.concat([df1, df2, df3]).groupby(level=0).mean()

avg = avg_of_columns(ml1,ml2,ml3)

print(avg)

plotMlProcessingTime = avg[["second", "median_processing_time_ms"]].rename(columns={"median_processing_time_ms": "value"}).assign(Metric="Median Processing Time")


fig, ax = plt.subplots(figsize=FIGSIZE_ACM_SMALL, constrained_layout=True)
sns.lineplot(
    data=plotMlProcessingTime,
    x="second",
    y="value",
    hue="Metric",
    style="Metric",
    markers=True,
    dashes=True,
    ax=ax
)

ax.set_xlabel("Time [s]")
ax.set_ylabel("Median Processing Time [s]")
ax.set_ylim(0, 60)

fig.savefig("plots/ml-processing-time.pdf", bbox_inches=False)
plt.close()

     second  median_processing_time_ms
0  .0.00000                  .23.33333
1 .10.00000                  .25.66667
2 .19.66667                  .26.66667
3 .30.00000                  .27.00000
4 .39.66667                  .27.00000
5 .49.00000                  .26.33333
6 .58.33333                  .29.00000


In [27]:

# 422.8611111111111
# 2047.0035460992908
# 12139.083333333332


plot1  = pd.DataFrame ({
    "Message Size": [1, 2, 3],
    "Median Processing Time": [0.4228611111111111, 2.0470035460992908, 12.139083333333332]
}).assign(Metric="Processing Times")

fig, ax = plt.subplots(figsize=FIGSIZE_ACM_SMALL, constrained_layout=True)

sns.lineplot(
    data=plot1,
    x="Message Size",
    y="Median Processing Time",
    hue="Metric",
    style="Metric",
    markers=True,
    dashes=True,
    ax=ax
)

ax.set_ylabel("Median Processing Time [s]")
ax.set_xlabel("Message Size [KB]")
ax.set_ylim(0, 30)
ax.set_xlim(0.5, 3.5)
ax.set_xticks([1,2,3])
ax.set_xticklabels(["$1$", "$10$", "$50$"])

handles, labels = ax.get_legend_handles_labels()

fig.savefig("plots/rps_impact_on_processing_time.pdf", bbox_inches=False)
plt.close()

In [32]:
# Ingestion Service Clean (new!)

def parse_csv_ingestion_service(filename: str) -> pd.DataFrame:
    def parse(data: list[str]) -> pd.DataFrame:
        return pd.DataFrame(data, columns=["message_id", "status_code", "t_send", "t_received"])
    data = get_csv_as_list(filename)
    return parse(data)

def transform_columns_to_numerics_is(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["message_id"] = pd.to_numeric(df["message_id"])
    df["status_code"] = pd.to_numeric(df["status_code"])
    df["t_send"] = pd.to_numeric(df["t_send"])
    df["t_received"] = pd.to_numeric(df["t_received"])
    return df

def remove_rampup_rampdown(df: pd.DataFrame, ramp_up_s: int = 15, ramp_down_s: int = 5) -> pd.DataFrame:
    t_start = df["t_send"].min()
    t_end   = df["t_send"].max()
    cutoff_start = t_start + ramp_up_s   * 1_000_000_000
    cutoff_end   = t_end   - ramp_down_s * 1_000_000_000
    return df[(df["t_send"] >= cutoff_start) & (df["t_send"] <= cutoff_end)]

def median_response_time(df: pd.DataFrame) -> float:
    ok = df[df["status_code"] == 200].copy()
    ok["response_time"] = (ok["t_received"] - ok["t_send"]) / 1_000_000

    median = ok["response_time"].median()
    return median

def process_ingestion_service(filename: str):
    df = parse_csv_ingestion_service(filename)
    df = transform_columns_to_numerics_is(df)
    df = remove_rampup_rampdown(df)
    median = median_response_time(df)
    return median

def process_ingestion_service_folder(folder: str) -> dict[str, float]:
    from pathlib import Path
    results = {}
    folder_path = Path(folder)

    for csv_file in sorted(folder_path.glob("*.csv")):
        median = process_ingestion_service(str(csv_file))
        results[csv_file.name] = median

    return results



def build_pivot_table(medians: dict[str, float]) -> pd.DataFrame:
    import re
    records = []
    for fname, median in medians.items():
        m = re.match(r'(\d+kb)_(\d+RPS)', fname)
        if m:
            records.append({"size": m.group(1), "rps": m.group(2), "median": median})
    return (pd.DataFrame(records)
              .groupby(["size", "rps"])["median"]
              .mean()
              .unstack("rps")
              .reindex(["1kb", "10kb", "25kb", "50kb"])
              [["1RPS", "10RPS", "25RPS", "50RPS"]])

medians = process_ingestion_service_folder("measurements/ingestion-service/")

pivot = build_pivot_table(medians)
pivot

rps,1RPS,10RPS,25RPS,50RPS
size,,,,
1kb,.4.17669,.3.64746,.2.94863,.3.21535
10kb,.5.36259,.4.18649,.4.17918,.3.91805
25kb,.6.81933,.4.85929,.5.15287,.4.55269
50kb,.8.02468,.6.99471,.5.62964,.6.23846
